# S03E04 — Negotiations (Tool Server)

Agent centrali szuka miast sprzedających podzespoły do turbiny wiatrowej.
Udostępniamy mu jedno narzędzie `/api/search` które:
1. Przyjmuje opis przedmiotu w języku naturalnym
2. Dopasowuje go do katalogu (keyword + LLM)
3. Zwraca listę miast sprzedających ten przedmiot

**Wymagania**: serwer Flask (uruchamiany inline w wątku) + publiczny tunel (serveo.net)

In [ ]:
import os, requests, time, re, csv, threading
from collections import defaultdict
from dotenv import load_dotenv, find_dotenv
from flask import Flask, request, jsonify
from anthropic import Anthropic

load_dotenv(find_dotenv())
API_KEY    = os.getenv("AI_DEVS_API_KEY")
VERIFY_URL = "https://hub.ag3nts.org/verify"

print("Konfiguracja OK.")

## 1. Pobierz dane CSV

In [ ]:
BASE = "https://hub.ag3nts.org/dane/s03e04_csv/"
for fname in ["cities.csv", "connections.csv", "items.csv"]:
    path = os.path.join(os.getcwd(), fname)
    if not os.path.exists(path):
        r = requests.get(BASE + fname)
        with open(path, "wb") as f:
            f.write(r.content)
        print(f"Downloaded {fname}")
    else:
        print(f"{fname} already exists")

## 2. Uruchom serwer Flask inline + tunel serveo

Serwer udostępnia endpoint `/api/search` z logiką:
- keyword pre-filter po katalogu items.csv
- Claude Haiku do disambiguacji gdy wiele wyników
- lookup miast z connections.csv

In [ ]:
# Wczytaj dane
DATA_DIR = os.getcwd()

items = []
with open(os.path.join(DATA_DIR, "items.csv")) as f:
    items = list(csv.DictReader(f))

item_by_code = {it["code"]: it["name"] for it in items}

city_by_code = {}
with open(os.path.join(DATA_DIR, "cities.csv")) as f:
    city_by_code = {c["code"]: c["name"] for c in csv.DictReader(f)}

item_cities = defaultdict(set)
with open(os.path.join(DATA_DIR, "connections.csv")) as f:
    for row in csv.DictReader(f):
        item_cities[row["itemCode"]].add(row["cityCode"])

claude = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

print(f"Załadowano {len(items)} przedmiotów, {len(city_by_code)} miast")

In [ ]:
# Logika wyszukiwania

def normalize(text: str) -> str:
    import re
    return re.sub(r"[^a-z0-9ąćęłńóśźż\s/.]", "", text.lower())


def keyword_search(query: str, top_n: int = 30) -> list[dict]:
    """Scoruj przedmioty po overlap słów kluczowych."""
    query_norm = normalize(query)
    words = [w for w in query_norm.split() if len(w) > 1]
    scored = []
    for item in items:
        name_norm = normalize(item["name"])
        score = sum(2 if len(w) > 3 else 1 for w in words if w in name_norm)
        if score > 0:
            scored.append((score, item))
    scored.sort(key=lambda x: -x[0])
    return [it for _, it in scored[:top_n]]


def match_with_llm(query: str, candidates: list[dict]) -> list[str]:
    """Claude Haiku wybiera najlepszy przedmiot z kandydatów."""
    if not candidates:
        return []
    catalog = "\n".join(f"{it['code']}|{it['name']}" for it in candidates)
    resp = claude.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=100,
        messages=[{
            "role": "user",
            "content": (
                f"Product catalog:\n{catalog}\n\n"
                f"User needs: \"{query}\"\n\n"
                "Return ONLY the single best matching item code. "
                "Match voltage, power, capacity etc. if specified. "
                "One code, nothing else."
            ),
        }],
    )
    code = resp.content[0].text.strip().split("\n")[0].strip()
    return [code] if code in item_by_code else []


def find_item(query: str) -> list[str]:
    """Znajdź kody przedmiotów dla zapytania."""
    query_upper = query.strip().upper()
    for code in item_by_code:
        if code in query_upper:
            return [code]
    candidates = keyword_search(query)
    if not candidates:
        return []
    if len(candidates) == 1:
        return [candidates[0]["code"]]
    try:
        codes = match_with_llm(query, candidates[:30])
        if codes:
            return codes
    except Exception:
        pass
    return [candidates[0]["code"]]


print("Logika wyszukiwania gotowa.")

In [ ]:
# Flask app + uruchomienie w wątku

app = Flask(__name__)


@app.route("/api/search", methods=["POST"])
def search():
    """Wyszukaj przedmioty i zwróć miasta w których są dostępne."""
    data = request.json or {}
    query = data.get("params", "")
    app.logger.info(f"QUERY: {query}")

    if not query or len(query.strip()) < 2:
        return jsonify({"output": "Provide an item description to search."})

    codes = find_item(query)

    if not codes:
        return jsonify({"output": "No matching items. Try different keywords."})

    parts = []
    for code in codes[:2]:
        name = item_by_code.get(code, "?")
        city_codes = item_cities.get(code, set())
        city_names = sorted(city_by_code.get(c, c) for c in city_codes)
        if city_names:
            parts.append(f"{name} [{code}]: {', '.join(city_names)}")
        else:
            parts.append(f"{name} [{code}]: unavailable")

    output = "\n".join(parts)
    if len(output.encode("utf-8")) > 495:
        output = output.encode("utf-8")[:495].decode("utf-8", errors="ignore")

    app.logger.info(f"RESPONSE: {output}")
    return jsonify({"output": output})


# Start Flask w wątku daemon (zatrzyma się gdy notebook zostanie zamknięty)
flask_thread = threading.Thread(
    target=lambda: app.run(port=5001, debug=False, use_reloader=False),
    daemon=True
)
flask_thread.start()
time.sleep(2)

# Test serwera lokalnie
try:
    r = requests.post("http://localhost:5001/api/search",
                      json={"params": "turbina wiatrowa 48V"})
    print(f"Server OK: {r.json()['output'][:80]}")
except Exception as e:
    print(f"Server error: {e}")

In [ ]:
# Uruchom tunel SSH do serveo.net
import subprocess

tunnel_proc = subprocess.Popen(
    ["ssh", "-o", "StrictHostKeyChecking=no", "-o", "ServerAliveInterval=60",
     "-R", "80:localhost:5001", "serveo.net"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
time.sleep(5)

# Odczytaj URL tunelu
output = tunnel_proc.stdout.readline().decode().strip()
print(f"Tunnel output: {output}")

tunnel_url = re.search(r'(https://[^\s]+)', output)
if tunnel_url:
    TUNNEL_URL = tunnel_url.group(1)
    print(f"Tunnel URL: {TUNNEL_URL}")
else:
    output2 = tunnel_proc.stdout.readline().decode().strip()
    tunnel_url = re.search(r'(https://[^\s]+)', output2)
    TUNNEL_URL = tunnel_url.group(1) if tunnel_url else "ERROR"
    print(f"Tunnel URL: {TUNNEL_URL}")

In [ ]:
# Test tunelu
r = requests.post(f"{TUNNEL_URL}/api/search",
                  json={"params": "turbina wiatrowa 48V"})
print(f"Tunnel test: {r.json()}")

## 3. Zarejestruj narzędzie w centrali

In [ ]:
resp = requests.post(VERIFY_URL, json={
    "apikey": API_KEY,
    "task": "negotiations",
    "answer": {
        "tools": [
            {
                "URL": f"{TUNNEL_URL}/api/search",
                "description": (
                    "Search for items available in cities. "
                    "Send the item name or description in Polish or English "
                    "as the params field (e.g. 'turbina wiatrowa 48V' or 'akumulator 48V 150Ah'). "
                    "Returns the item name and list of cities where it can be purchased. "
                    "Use this tool to find which cities sell a specific item."
                ),
            }
        ]
    },
})
print(f"Registration: {resp.json()}")

## 4. Czekaj na wynik i sprawdź flagę

In [ ]:
# Poczekaj ~45 sekund na agenta, potem sprawdz wynik
print("Czekam 45s na agenta...")
time.sleep(45)

resp = requests.post(VERIFY_URL, json={
    "apikey": API_KEY,
    "task": "negotiations",
    "answer": {"action": "check"},
})
result = resp.json()
print(f"Result: {result}")

if result.get("code") != 0:
    print("Jeszcze nie gotowe, sprobuj ponownie za chwile...")

In [ ]:
# Cleanup - zatrzymaj tunel (serwer Flask zatrzyma się sam gdy kernel zostanie zamknięty)
tunnel_proc.terminate()
print("Tunel zatrzymany.")